# [수학 Retreat #7] 피타고라스 정리: 유클리드의 증명과 3D 공간 거리 계측

이 주피터 노트북은 중학교 2학년 과정의 **피타고라스 정리** 단원을 파이썬 시각화 도구를 통해 다각도로 탐구하는 실습 교재입니다.
직각의 올곧음이 빚어내는 제곱의 합의 조화($a^2 + b^2 = c^2$)와 그것의 3차원 공간적 확장을 배웁니다.

### 🟢 본 실습에서 다룰 두 가지 주제:
1. **유클리드 증명 애니메이터 (`euclidean_proof_animator`)**: 기하학 역사상 가장 아름답고 엄밀하다고 평가받는 유클리드의 증명 과정을 슬라이더를 부드럽게 넘기며 **평행이동(등적변형)과 90도 회전**으로 면적이 같아짐을 실시간으로 확인하는 애니메이션 도구.
2. **3D 공간 거리 계측기 (`distance_detector_3d`)**: 3차원 가상 좌표계 위에 두 점을 띄우고 조작할 때, 두 점 사이의 공간 거리($d = \sqrt{dx^2 + dy^2 + dz^2}$)가 어떻게 피타고라스 정리의 입체 확장으로 연산되는지 눈으로 확인하는 3D 시뮬레이터.

**⚠️ 안내:** 본 파일은 파이썬 초보자도 코드 한 줄 한 줄의 기하학적·수학적 연산 원리를 이해할 수 있도록 **세포 단위의 멀티셀**로 쪼개어 구성했으며 상세한 한글 주석을 부착했습니다.


## 1. 환경 준비 및 필수 라이브러리 설치
본 실습에서는 2D/3D 벡터 그래픽 연산을 위해 `numpy`, 인터랙티브 시뮬레이션 구현을 위해 `plotly`, 그리고 제어 패널 구성을 위해 `ipywidgets`를 활용합니다.
아래 셀을 실행하여 라이브러리를 먼저 준비합니다.


In [ ]:
# !pip install 명령어는 주피터 노트북 내에서 터미널의 패키지 매니저를 호출하여 패키지를 설치하게 하는 매직 명령어입니다.
# -q (quiet) 옵션을 붙여 설치 메시지를 생략하고 설치합니다.
%pip install -q numpy plotly ipywidgets


### 1.2 모듈 로드 (Import)
기하 좌표의 회전 변환 및 3차원 공간 렌더링에 필요한 파이썬 모듈들을 메모리에 로드합니다.


In [ ]:
# numpy: 삼각함수, 행렬 연산 및 3D 점 좌표 계산의 기초가 되는 수학 라이브러리입니다.
import numpy as np

# plotly.graph_objects: 3차원 공간 좌표축 및 면적 변형 애니메이션을 유연하게 그리는 그래프 빌더 모듈입니다.
import plotly.graph_objects as go

# ipywidgets: 증명 단계를 제어할 슬라이더 바와 3D 점들의 x, y, z 제어 슬라이더반 구성을 돕는 라이브러리입니다.
import ipywidgets as widgets

# IPython.display: 수치 리포트 대시보드를 주피터 노트북 내에 HTML 박스로 표현하기 위해 로드합니다.
from IPython.display import display, HTML


---
## 2. 실습 1: 유클리드의 피타고라스 정리 증명 애니메이터 (`euclidean_proof_animator`)
유클리드는 직각삼각형 $ABC$ (각 $C = 90^\circ$)의 세 변에 각각 정사각형을 그린 뒤, 다음과 같은 흐름으로 왼쪽 정사각형의 넓이가 빗변 밑의 왼쪽 직사각형의 넓이와 같아짐을 보였습니다.

### 📐 유클리드 증명 단계의 4가지 프레임 기하학:
1. **0단계 (출발점)**: 왼쪽 정사각형의 정확히 절반 면적을 가지는 삼각형 $GAC$에서 출발합니다.
2. **1단계 (등적변형 1)**: 꼭짓점 C를 밑변 $GA$와 평행한 방향을 따라 B로 평행이동시킵니다. 밑변과 높이가 같으므로 면적이 동일한 삼각형 $GAB$가 완성됩니다.
3. **2단계 (회전변환)**: 삼각형 $GAB$를 꼭짓점 A를 중심축으로 삼아 시계방향으로 $90^\circ$ 회전시킵니다. 두 변과 끼인각이 같으므로(SAS 합동) 면적이 완벽히 같은 삼각형 $CAH$로 변형됩니다.
4. **3단계 (등적변형 2)**: 꼭짓점 C를 밑변 $AH$와 평행한 수선 방향을 따라 수선의 발 $P_0$로 평행이동시킵니다. 밑변과 높이가 같으므로 면적이 동일하며 빗변 밑 왼쪽 직사각형 절반에 해당하는 삼각형 $P_0AH$가 완성됩니다.

이 보간 흐름을 수학적으로 정합성 있게 좌표계로 구현하기 위한 렌더링 함수를 작성하겠습니다.


In [ ]:
def get_euclidean_proof_points(a, b, s_val):
    """
    직각을 끼고 있는 두 변의 길이 a(밑변), b(높이)와 애니메이션 진행도 s_val (0.0 ~ 3.0)을 받아
    실시간으로 보간/회전 변환이 완료된 삼각형의 꼭짓점 4개(닫힌 루프) 좌표를 계산합니다.
    """
    # 1. 고정 기준 좌표계 정의 (직각 꼭짓점 C를 원점 [0, 0]으로 설정)
    C = np.array([0.0, 0.0])
    A = np.array([0.0, float(b)])
    B = np.array([float(a), 0.0])
    
    # 2. 각 변 바깥쪽에 그린 정사각형들의 꼭짓점
    G = np.array([-b, b])  # 왼쪽 정사각형의 좌상단 꼭짓점
    F = np.array([-b, 0.0]) # 왼쪽 정사각형의 좌하단 꼭짓점
    
    # 빗변 AB 아래쪽 정사각형의 꼭짓점 H(A로부터 수직 이동), K(B로부터 수직 이동)
    H = A + np.array([b, a])
    K = B + np.array([b, a])
    
    # 3. 원점 C에서 빗변 AB에 내린 수선의 발 P0 좌표 연산
    c_sq = a**2 + b**2
    P0 = np.array([a * b**2 / c_sq, a**2 * b / c_sq])
    
    # 4. 각 단계별 핵심 삼각형들의 꼭짓점 상태 세트 정의
    # 닫힌 다각형을 그리기 위해 꼭짓점 순서는 [1, 2, 3, 1] 형태로 회폐쇄합니다.
    p0 = np.array([G, A, C, G])   # 0단계: GAC (정사각형 넓이의 1/2)
    p1 = np.array([G, A, B, G])   # 1단계: GAB (꼭짓점 C가 B로 평행이동, 등적변형 1)
    p2 = np.array([C, A, H, C])   # 2단계: CAH (합동 회전 변환 완료 상태)
    p3 = np.array([P0, A, H, P0])  # 3단계: P0AH (꼭짓점 C가 P0로 평행이동, 등적변형 2)
    
    # 5. 진행 지수 s_val의 범위에 따라 대수적 변환 적용
    if s_val < 1.0:
        # [구간 0 ~ 1]: GAC -> GAB로 단순 평행이동 (선형 보간)
        t = s_val
        pts = (1 - t) * p0 + t * p1
        
    elif s_val < 2.0:
        # [구간 1 ~ 2]: GAB -> CAH로 꼭짓점 A를 중심축 삼아 시계방향 90도 회전
        # 단순 선형 보간을 쓰면 회전 중에 크기가 작아지는 왜곡이 생기므로, 회전 행렬을 정밀 적용합니다.
        t = s_val - 1.0
        angle = np.radians(-90.0 * t) # t가 0에서 1로 갈 때 0도에서 -90도(시계방향)로 회전
        rot_matrix = np.array([
            [np.cos(angle), -np.sin(angle)],
            [np.sin(angle), np.cos(angle)]
        ])
        
        # 꼭짓점 G와 B를 A를 기준으로 회전시킵니다.
        # 회전 후 꼭짓점 G_rot = A + rot * (G - A)
        # 회전 후 꼭짓점 B_rot = A + rot * (B - A)
        G_rot = A + rot_matrix.dot(G - A)
        B_rot = A + rot_matrix.dot(B - A)
        A_rot = A.copy()
        
        pts = np.array([G_rot, A_rot, B_rot, G_rot])
        
    else:
        # [구간 2 ~ 3]: CAH -> P0AH로 수선 방향 평행이동 (선형 보간)
        t = s_val - 2.0
        pts = (1 - t) * p2 + t * p3
        
    return pts, C, A, B, G, F, H, K, P0


### 2.2 실시간 등적변형 및 기하 성질 리포트 대시보드
슬라이더로 조절되는 현재 애니메이션 단계(0.0 ~ 3.0)에 상응하는 기하학적 증명 과정의 수학적 해설을 실시간으로 안내해 주는 HTML 대시보드 함수를 작성합니다.


In [ ]:
def build_proof_status_html(s_val):
    """
    현재 s_val 수치에 맞추어 유클리드 기하학 증명 단계를 친절히 서술해 주는 HTML 대시보드를 생성합니다.
    """
    if s_val == 0.0:
        step_title = "🟢 0단계: 출발 기하 정의"
        step_desc = "왼쪽 정사각형의 넓이의 1/2을 가진 <b>삼각형 GAC</b>에서 출발합니다."
    elif 0.0 < s_val < 1.0:
        step_title = "[/] 0 -> 1단계: 등적변형 진행 중 (평행이동)"
        step_desc = "꼭짓점 C가 밑변 GA와 평행한 방향을 따라 B로 평행이동합니다. 밑변과 높이가 유지되므로 <b>삼각형의 넓이는 완벽히 일정</b>합니다."
    elif s_val == 1.0:
        step_title = "🟢 1단계: 등적변형 1 완료 (삼각형 GAB)"
        step_desc = "정사각형의 넓이의 절반을 그대로 보존한 <b>삼각형 GAB</b>가 만들어졌습니다."
    elif 1.0 < s_val < 2.0:
        step_title = "[/] 1 -> 2단계: 꼭짓점 A 기준 시계방향 90도 회전 중"
        step_desc = "꼭짓점 A를 고정축 삼아 시계방향으로 90도 미끄러지며 회전합니다. (두 변의 길이와 끼인각이 같으므로 <b>SAS 합동</b> 관계를 증명)"
    elif s_val == 2.0:
        step_title = "🟢 2단계: 합동 회전 완료 (삼각형 CAH)"
        step_desc = "삼각형 GAB와 넓이가 한 치의 오차도 없이 완전히 합동인 <b>삼각형 CAH</b>가 빗변 정사각형 영역에 안착했습니다."
    elif 2.0 < s_val < 3.0:
        step_title = "[/] 2 -> 3단계: 등적변형 진행 중 (평행이동)"
        step_desc = "꼭짓점 C가 밑변 AH와 평행한 수선 가이드를 따라 수선의 발 P0로 미끄러져 평행이동합니다. 넓이는 역시 보존됩니다."
    else:
        step_title = "🟢 3단계: 등적변형 2 완료 (삼각형 P0AH)"
        step_desc = "빗변 아래 왼쪽 직사각형 면적의 절반에 정확히 도달했습니다. 결과적으로 <b>(왼쪽 정사각형 면적의 1/2) = (왼쪽 직사각형 면적의 1/2)</b>이 증명되었습니다!" 
        
    html_panel = f"""
    <div style='background: rgba(255, 255, 255, 0.85);
                backdrop-filter: blur(8px);
                border: 1.5px solid rgba(11, 121, 208, 0.25);
                border-radius: 14px; padding: 18px; max-width: 650px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.05);'>
        <h4 style='margin: 0 0 8px 0; color: #0B79D0;'>📖 유클리드 등적 증명 실시간 해설</h4>
        <div style='font-size: 1.05em; font-weight: bold; color: #1E2937;'>{step_title}</div>
        <p style='margin: 6px 0 0 0; font-size: 0.92em; color: #475569;'>{step_desc}</p>
    </div>
    """
    return html_panel


### 2.3 Plotly 2D 유클리드 기하 렌더러 설계
배경이 될 직각삼각형, 세 변 위의 3개 정사각형, 꼭짓점 C에서 내린 수선 기둥선, 그리고 실시간 변형되는 삼각형 트레이스를 캔버스 상에 그리는 함수를 구축합니다.
변형되는 핵심 삼각형은 눈에 잘 띄도록 반투명 빨간색 컬러(`rgba(225, 29, 72, 0.5)`)로 렌더링합니다.


In [ ]:
def draw_euclidean_proof_scene(side_a, side_b, s_value):
    """
    입력받은 기하 크기와 슬라이더 위치에 부합하는 유클리드 증명 작도 화면을 그립니다.
    """
    # 1. 보간된 정점 좌표 및 전체 작도점 좌표 연산
    interp_pts, C, A, B, G, F, H, K, P0 = get_euclidean_proof_points(side_a, side_b, s_value)
    
    # 2. 주피터 상단에 실시간 단계 해설 대시보드 표출
    display(HTML(build_proof_status_html(s_value)))
    
    traces = []
    
    # 3. 바탕 도형 그리기
    # 1) 원조 직각삼각형 ABC (검은색 굵은 실선)
    traces.append(go.Scatter(
        x=[A[0], B[0], C[0], A[0]], y=[A[1], B[1], C[1], A[1]],
        mode='lines',
        line=dict(color='#1F2937', width=4),
        name='직각삼각형 ABC'
    ))
    
    # 2) 왼쪽 높이변 위 정사각형 ACFG (연한 하늘색 점선)
    traces.append(go.Scatter(
        x=[A[0], G[0], F[0], C[0], A[0]], y=[A[1], G[1], F[1], C[1], A[1]],
        mode='lines',
        line=dict(color='#0B79D0', width=1.5, dash='dash'),
        fill='toself', fillcolor='rgba(11, 121, 208, 0.04)', # 연하게 채우기
        name='정사각형 ACFG (b²)'
    ))
    
    # 3) 밑바닥변 위 정사각형 BCED (연한 초록색 점선)
    D_pt = np.array([0.0, -side_a])
    E_pt = np.array([side_a, -side_a])
    traces.append(go.Scatter(
        x=[C[0], D_pt[0], E_pt[0], B[0], C[0]], y=[C[1], D_pt[1], E_pt[1], B[1], C[1]],
        mode='lines',
        line=dict(color='#2FA85D', width=1.5, dash='dash'),
        fill='toself', fillcolor='rgba(47, 168, 93, 0.04)',
        name='정사각형 BCED (a²)'
    ))
    
    # 4) 빗변 위 정사각형 ABKH (회색 점선)
    traces.append(go.Scatter(
        x=[A[0], B[0], K[0], H[0], A[0]], y=[A[1], B[1], K[1], H[1], A[1]],
        mode='lines',
        line=dict(color='#6B7280', width=1.5, dash='dash'),
        name='정사각형 ABKH (c²)'
    ))
    
    # 5) C에서 내린 면적 분할 수선 기둥 (Altitude Line)
    # 수선의 발 P0에서 반대편 변 HK의 점 P1까지 잇는 가이드선 그리기
    P1 = P0 + np.array([side_b, side_a])
    traces.append(go.Scatter(
        x=[C[0], P0[0], P1[0]], y=[C[1], P0[1], P1[1]],
        mode='lines',
        line=dict(color='#475569', width=2, dash='dot'), # 짙은 쥐색 점선
        name='수선 가이드라인'
    ))
    
    # 4. 실시간 변형되는 유클리드 증명 삼각형 (강렬한 로즈 핑크 채우기)
    traces.append(go.Scatter(
        x=interp_pts[:, 0], y=interp_pts[:, 1],
        mode='lines+markers',
        line=dict(color='#E11D48', width=4.5),
        marker=dict(size=8, color='#E11D48'),
        fill='toself', fillcolor='rgba(225, 29, 72, 0.45)', # 45%의 반투명 채우기로 기하 겹침 투과
        name='변형 증명 삼각형'
    ))
    
    # 레이아웃 정의 (x-y 1:1 비율 균등 등배율 고정 필수)
    bound = max(side_a, side_b) * 2.2
    layout = go.Layout(
        title=dict(text='<b>유클리드 피타고라스 등적변형 시각 시뮬레이터</b>', x=0.5),
        xaxis=dict(range=[-bound, bound], gridcolor='#F1F5F9', zeroline=True, zerolinecolor='#CBD5E1'),
        yaxis=dict(range=[-bound, bound], gridcolor='#F1F5F9', scaleanchor='x', scaleratio=1.0, zeroline=True, zerolinecolor='#CBD5E1'),
        plot_bgcolor='white',
        width=650, height=650,
        showlegend=True,
        legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center')
    )
    
    fig = go.Figure(data=traces, layout=layout)
    fig.show()


### 2.4 유클리드 기하 변형 슬라이더 기동
삼각형의 한 변의 길이를 결정하는 직각 두 변 $a, b$ 크기 슬라이더와,
증명의 진행도를 제어하는 **진행 지수( s_value, 0.0에서 3.0)** 슬라이더를 구성하여 실행합니다.
진행 슬라이더를 $0.0 \rightarrow 1.0 \rightarrow 2.0 \rightarrow 3.0$으로 점진적으로 조절하면서, 붉은색 삼각형이 미끄러지고 회전하며 직사각형으로 안착하는 기하학적 등적 변형을 직접 체험해 보세요.


In [ ]:
# 직각 두 변의 길이를 지정하는 슬라이더 생성 (각각 2에서 5까지)
slider_a = widgets.FloatSlider(value=3.0, min=2.0, max=5.0, step=0.1, description='밑변 a의 길이:')
slider_b = widgets.FloatSlider(value=4.0, min=2.0, max=5.0, step=0.1, description='높이 b의 길이:')

# 증명 애니메이션 진행 슬라이더 생성 (0에서 3까지 0.05 단위 부드러운 제어)
slider_s = widgets.FloatSlider(
    value=0.0, min=0.0, max=3.0, step=0.05,
    description='애니메이션 단계:',
    continuous_update=True # 드래그 시 실시간으로 등적변형이 화면에 연동 반영됩니다.
)

controls_box = widgets.VBox([
    widgets.HTML("<h4 style='color:#0B79D0; margin:0;'>📐 직각삼각형 규격 설정</h4>"),
    slider_a, slider_b,
    widgets.HTML("<h4 style='color:#E11D48; margin:0;'>🎬 증명 변형 제어 슬라이더 (0 ~ 3)</h4>"),
    slider_s
])

output_proof = widgets.interactive_output(
    draw_euclidean_proof_scene,
    {'side_a': slider_a, 'side_b': slider_b, 's_value': slider_s}
)

display(widgets.HBox([controls_box, output_proof]))


---
## 3. 실습 2: 3차원 유클리드 공간 거리 계측기 (`distance_detector_3d`)
피타고라스 정리의 진정한 위대함은 우리가 발을 딛고 숨 쉬는 3차원 공간 전체의 길이를 정량화하는 **유클리드 거리 공식**으로 완벽하게 연장된다는 데 있습니다.

### 3차원 거리 공식 유도 과정:
1. 3D 공간 상에 두 점 $P_1(x_1, y_1, z_1)$과 $P_2(x_2, y_2, z_2)$가 존재할 때, 각 축 방향의 차이 $dx = x_2 - x_1$, $dy = y_2 - y_1$, $dz = z_2 - z_1$을 계산합니다.
2. 바닥면($xy$ 평면) 상에서의 투영 빗변의 거리 $d_{xy}$는 피타고라스 정리에 의해 다음과 같습니다:
   $$d_{xy}^2 = dx^2 + dy^2$$
3. 이 바닥면 빗변($d_{xy}$)과 높이축 차이($dz$)는 기하학적으로 완벽히 직각을 이루므로, 3D 최종 대각선 공간 빗변 거리 $d$는 다시 피타고라스 공식에 의해 결합됩니다:
   $$d^2 = d_{xy}^2 + dz^2 = dx^2 + dy^2 + dz^2$$
   $$d = \sqrt{dx^2 + dy^2 + dz^2}$$

두 점의 x, y, z 좌표 슬라이더들을 조작할 때 실시간으로 이 3차원 뼈대를 추적하여 Plotly 3D 공간 상에 입체적으로 그려주는 시뮬레이터 함수를 정의합니다.


In [ ]:
def render_3d_distance_detector(p1_x, p1_y, p1_z, p2_x, p2_y, p2_z):
    """
    두 점의 3D 좌표를 받아 3D 피타고라스 계측 가이드 뼈대와 실시간 거리를 렌더링합니다.
    """
    # 1. 두 점의 numpy 배열 좌표 벡터 생성
    P1 = np.array([p1_x, p1_y, p1_z])
    P2 = np.array([p2_x, p2_y, p2_z])
    
    # 2. 각 축 방향 변위 계산
    dx = p2_x - p1_x
    dy = p2_y - p1_y
    dz = p2_z - p1_z
    
    # 3. 바닥 투영 거리 및 최종 3D 공간 거리 연산 (피타고라스 3차원 확장)
    d_xy = np.sqrt(dx**2 + dy**2)
    d_xyz = np.sqrt(dx**2 + dy**2 + dz**2)
    
    # 4. 실시간 수치 정보 HTML 대시보드 리포팅 표출
    dashboard_html = f"""
    <div style='background: linear-gradient(135deg, rgba(11,121,208,0.06), rgba(47,168,93,0.06));
                padding: 18px; border-radius: 12px; border: 1.5px solid rgba(11,121,208,0.2);
                font-family: sans-serif; max-width: 600px; margin-bottom: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.05);'>
        <h4 style='margin: 0 0 8px 0; color: #0B79D0;'>📐 3D 공간 피타고라스 실시간 거리 계측</h4>
        <div style='display: flex; justify-content: space-around; text-align: center; font-size: 0.92em;'>
            <div>
                <span style='color: #666;'>1. 바닥 성분 변위 (dx, dy)</span>
                <h3 style='margin: 4px 0 0 0; color: #475569;'>dx={abs(dx):.1f} | dy={abs(dy):.1f}</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 15px;'>
                <span style='color: #666;'>2. 바닥 평면 빗변 (d_xy)</span>
                <h3 style='margin: 4px 0 0 0; color: #0B79D0;'>d_xy = {d_xy:.2f}</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 15px;'>
                <span style='color: #666;'>3. 공간 대각선 거리 (d_xyz)</span>
                <h3 style='margin: 4px 0 0 0; color: #2FA85D; font-weight: bold;'>d_xyz = {d_xyz:.2f}</h3>
            </div>
        </div>
    </div>
    """
    display(HTML(dashboard_html))
    
    traces = []
    
    # 5. Plotly 3D 성분 렌더링
    # 1) 두 점 P1, P2 점 마커
    traces.append(go.Scatter3d(
        x=[p1_x, p2_x], y=[p1_y, p2_y], z=[p1_z, p2_z],
        mode='markers+text',
        marker=dict(size=8, color=['#0B79D0', '#E11D48'], symbol='circle', line=dict(color='white', width=1)),
        text=['P1 (출발점)', 'P2 (대상점)'], textposition='top center',
        name='계측 좌표점들'
    ))
    
    # 2) 공간 대각선 연결선 (실제 거리 d_xyz, 빨간색 굵은 실선)
    traces.append(go.Scatter3d(
        x=[p1_x, p2_x], y=[p1_y, p2_y], z=[p1_z, p2_z],
        mode='lines',
        line=dict(color='#E11D48', width=5),
        name='공간 최단 거리 (d_xyz)'
    ))
    
    # 3) 바닥 투영 직각삼각형 경로 뼈대선 그리기
    # 경로: P1 -> (p2_x, p2_y, p1_z)[바닥 평면 빗변 끝점] -> P2
    mid_pt = np.array([p2_x, p2_y, p1_z])
    traces.append(go.Scatter3d(
        x=[p1_x, mid_pt[0], p2_x],
        y=[p1_y, mid_pt[1], p2_y],
        z=[p1_z, mid_pt[2], p2_z],
        mode='lines+markers',
        line=dict(color='#2FA85D', width=3, dash='dash'), # 초록색 점선 경로
        marker=dict(size=4, color='#2FA85D'),
        name='바닥면 직각삼각형 경로'
    ))
    
    # 4) 축 성분별 개별 수직선분 (dx, dy 가이드)
    traces.append(go.Scatter3d(
        x=[p1_x, p2_x, p2_x],
        y=[p1_y, p1_y, p2_y],
        z=[p1_z, p1_z, p1_z],
        mode='lines',
        line=dict(color='#94A3B8', width=2, dash='dot'), # 옅은 회색 점선
        name='dx / dy 축성분선',
        hoverinfo='skip'
    ))
    
    # 6. 3D 차트 레이아웃 스타일 설정
    layout = go.Layout(
        title=dict(text='<b>3차원 공간 피타고라스 계측기 시각화</b>', x=0.5),
        scene=dict(
            xaxis=dict(title='X축', range=[-10, 10], backgroundcolor="rgb(248, 250, 252)", showbackground=True),
            yaxis=dict(title='Y축', range=[-10, 10], backgroundcolor="rgb(248, 250, 252)", showbackground=True),
            zaxis=dict(title='Z축', range=[-10, 10], backgroundcolor="rgb(248, 250, 252)", showbackground=True),
            aspectratio=dict(x=1, y=1, z=1) # 3D 공간 1:1:1 직교 왜곡 없음 방지
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        width=750, height=600,
        showlegend=True,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig = go.Figure(data=traces, layout=layout)
    fig.show()


### 3.3 3D 공간 제어 위젯 조립 및 거리 계측기 실행
두 점 P1과 P2의 x, y, z 좌표(각각 -7부터 +7 범위)를 독립 제어하는 6개의 슬라이더 조작 패널을 조립하여 실행합니다.
마우스 드래그로 3차원 그리드를 동적으로 회전시켜 보며, 3D 공간 빗변이 바닥의 피타고라스 삼각형($d_{xy}$)과 높이축($dz$)의 결합으로 직관적으로 설명되는 3D 공간 피타고라스 기하를 통찰해 보세요.


In [ ]:
# 출발점 P1(x, y, z) 제어 슬라이더
slider_p1x = widgets.FloatSlider(value=-3.0, min=-7.0, max=7.0, step=0.2, description='P1_X 좌표:')
slider_p1y = widgets.FloatSlider(value=-2.0, min=-7.0, max=7.0, step=0.2, description='P1_Y 좌표:')
slider_p1z = widgets.FloatSlider(value=-1.0, min=-7.0, max=7.0, step=0.2, description='P1_Z 좌표:')

# 대상점 P2(x, y, z) 제어 슬라이더
slider_p2x = widgets.FloatSlider(value=4.0, min=-7.0, max=7.0, step=0.2, description='P2_X 좌표:')
slider_p2y = widgets.FloatSlider(value=3.0, min=-7.0, max=7.0, step=0.2, description='P2_Y 좌표:')
slider_p2z = widgets.FloatSlider(value=4.0, min=-7.0, max=7.0, step=0.2, description='P2_Z 좌표:')

# 위젯 VBox 정렬
controls_box = widgets.VBox([
    widgets.HTML("<h4 style='color:#0B79D0; margin:0;'>🔵 P1 (출발점) 좌표 제어</h4>"),
    slider_p1x, slider_p1y, slider_p1z,
    widgets.HTML("<h4 style='color:#E11D48; margin:0;'>🔴 P2 (대상점) 좌표 제어</h4>"),
    slider_p2x, slider_p2y, slider_p2z
])

# 슬라이더들과 3D 시각화 엔진 바인딩
output_scene = widgets.interactive_output(
    render_3d_distance_detector,
    {
        'p1_x': slider_p1x, 'p1_y': slider_p1y, 'p1_z': slider_p1z,
        'p2_x': slider_p2x, 'p2_y': slider_p2y, 'p2_z': slider_p2z
    }
)

# 가로 방향 HBox 레이아웃 배치 기동
display(widgets.HBox([controls_box, output_scene]))


---
## 4. 깊이 있는 기하학 사색을 위한 열린 질문 (Joshua를 위한 질문)

1. **면적의 등적변형과 보존 법칙**:
   유클리드 증명의 핵심은 직각삼각형 좌측 정사각형($b^2$)의 붉은 삼각형 영역이 그 형태(꼭짓점의 상대각도와 형상)를 완전히 일그러뜨리며 등적변형 및 회전을 거듭했음에도 불구하고, **최종 안착한 직사각형의 면적과 완벽한 동등성을 보존**한다는 사실에 있습니다.
   우리 조직이나 사업 모델의 외형이나 규격(삼각형의 모양)을 시대적 요구에 맞춰 격렬하게 변형(피벗, Reframing)할 때도, 결코 훼손되지 않고 보존해야 할 비즈니스의 **'본질적인 면적 가치(Core Value)'**는 무엇이며, 이를 기하학적 등적 변형에 빗대어 어떻게 서술할 수 있을까요?

2. **3D 가상 공간의 차원 확장성과 피타고라스**:
   2차원 평면의 피타고라스 $a^2 + b^2 = c^2$ 정리가 새로운 축인 Z축 차원을 추가했을 뿐인데 $d^2 = dx^2 + dy^2 + dz^2$ 라는 3차원 유클리드 공간 전체를 계측하는 초월적인 거리 공식으로 확장되었습니다.
   이처럼 나의 비즈니스나 삶에 새로운 '차원(새로운 가치관, 신기술, 혹은 새로운 비즈니스 모델 파트너)'을 한 단계 누적(확장)하여, 기존 시스템을 완전히 넘어서는 다차원적 비약(Scale-up)을 이뤄냈거나 기획 중인 부분이 있다면 기하학적 차원 확장에 빗대어 성찰해 보세요.
